# USR Phase in Higgs Inflation & the Low-$\ell$ CMB Power Deficit

**From background dynamics through $\mathcal{P}_{\mathcal{R}}(k)$ to the Sachs-Wolfe $C_\ell$**


## Motivation

Planck 2018 low-$\ell$ TT power shows a $\sim$10–20% deficit at $\ell \lesssim 30$ relative to the $\Lambda$CDM best-fit extrapolation. Standard slow-roll Higgs inflation predicts a slightly red-tilted spectrum ($n_s \approx 0.967$ at $N_* = 60$) that matches the high-$\ell$ data well but does not address the low-$\ell$ anomaly — the extrapolated power at large scales remains above the observed deficit. This has motivated the study of transient non-attractor phases — particularly Ultra-Slow Roll (USR) — that can locally suppress power on the largest scales.

This notebook walks step-by-step through the full calculation chain: background USR dynamics $\rightarrow$ the $\varepsilon_2$ plateau at $-6$ $\rightarrow$ a localized dip in $\mathcal{P}_{\mathcal{R}}(k)$ $\rightarrow$ Sachs-Wolfe $C_\ell$ $\rightarrow$ comparison with Planck 2018 binned data. A clear mapping between code variables and physical conventions is provided throughout.


## Convention check: code variables vs physics

| Quantity | Code variable | Physical formula | USR value |
|---|---|---|---|
| $\varepsilon_1$ | `epsH` | $-\dot H/H^2 = \tfrac12\dot\phi^2/H^2$ | $\rightarrow 0\ (\propto a^{-6})$ |
| $\eta_{\rm code}$ | `etaH` | $-\ddot\phi/(H\dot\phi)$ | $\approx +3$ |
| $\varepsilon_2$ | *(derived)* | $d\ln\varepsilon_1/dN = \dot\varepsilon_1/(\varepsilon_1H)$ | $\rightarrow -6$ |

**Notational context.** The cosmology literature historically overlaps three distinct
slow-roll parameter frameworks:

1. **Potential slow-roll** ($\epsilon_V$, $\eta_V \equiv M_P^2 V''/V$): Liddle, Lyth,
   Steinhardt & Turner (1990s). Applies strictly during monotonic slow-roll.
2. **Hubble slow-roll** ($\epsilon_H$, $\eta_H \equiv -\ddot\phi/(H\dot\phi) =
   \texttt{etaH}$): Hamilton-Jacobi formulation (mid-1990s). This is the convention
   used by the code.
3. **Horizon Flow** ($\epsilon_1$, $\epsilon_2$): Schwarz, Terrero-Escalante & Garc\'{\i}a
   (2001). Defined via hierarchical logarithmic derivatives of $H$;
   $\epsilon_1 = \epsilon_H$ and $\epsilon_2 = d\ln\epsilon_1/dN$.

All three converge at leading order in slow-roll ($\epsilon_1 \simeq \epsilon_V$) but
diverge significantly during non-attractor dynamics. A frequent point of confusion is
the assumption that the second-order parameters are equivalent: $\epsilon_2 \neq \eta_V
\neq \eta_H$.

**The literature statement $\eta = -6$ in USR refers to $\epsilon_2$,** not the code
variable `etaH`. The relation is

$$\varepsilon_2 = -2\,\eta_{\rm code} + 2\,\varepsilon_1$$

Throughout this notebook we use $\varepsilon_1$ (`epsH`) and $\varepsilon_2$ (derived
from $\varepsilon_1$ via numerical differentiation).

**Code units:** $x = \phi/M_P$, $y = d\phi/dT$, $z = H/(S\,M_P)$ with $S = 5\times10^{-5}$.
The ODE integration variable $T$ is physical time in code units.


In [ ]:
# ============================================================
# EDIT THESE PARAMETERS to test different configurations
# ============================================================

FIELD_CONFIG = dict(
    phi0=5.70,
    yi=-0.10,
    xi=15000.0,
    lam=0.13,
)

SR_CONFIG = dict(
    phi0=6.0,
    yi=-0.001,
    xi=15000.0,
    lam=0.13,
)

NUM_PARAMS = dict(
    N_star=60,
    k_min=1e-5,
    k_max=1.0,
    num_k=80,
    k_pivot_phys=0.05,
    As=2.1e-9,
    use_weighted=True,
)

print('Config loaded.')
print(f'USR: phi0={FIELD_CONFIG["phi0"]}, yi={FIELD_CONFIG["yi"]}, xi={FIELD_CONFIG["xi"]}')
print(f'SR:  phi0={SR_CONFIG["phi0"]}, yi={SR_CONFIG["yi"]}, xi={SR_CONFIG["xi"]}')


In [ ]:
import sys, os, json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

from models import HiggsModel
from inf_dyn_background import run_background_simulation, get_derived_quantities
from scripts.pspectrum_pipeline import run_pspectrum_pipeline, load_pspectrum, build_weighted_kgrid
from scripts.camb_wrapper import compute_cl_sw, compute_cl_sw_powerlaw
from scripts.planck_data import get_planck_data

plt.rcParams.update({'font.size': 12, 'axes.labelsize': 13, 'legend.fontsize': 10, 'figure.figsize': (12, 5)})

print('Imports ready.')


## Background: equations of motion

The Friedmann and Klein-Gordon equations in the Einstein frame ($M_P = 1$) are:

$$H^2 = \frac{1}{3}\Bigl(\frac12\dot\phi^2 + V(\phi)\Bigr),\qquad
\ddot\phi + 3H\dot\phi + V'(\phi) = 0$$

Higgs potential (high-field approximation):

$$V(\phi) \approx \frac{\lambda}{4\xi^2}\bigl(1-e^{-\alpha\phi}\bigr)^2,\quad
\alpha = \sqrt{2/3}$$

**Code mapping.** The ODE system uses the variables $[x, y, z, n]$:

$$\frac{dx}{dT} = y,\quad
\frac{dy}{dT} = -3zy - \frac{v_0}{S^2}\,\frac{df}{dx},\quad
\frac{dz}{dT} = -\frac12 y^2,\quad
\frac{dn}{dT} = z$$

The slow-roll parameters are then extracted from the numerical solution:

$$\varepsilon_1 = -\frac{\dot H}{H^2} = \frac12\frac{\dot\phi^2}{H^2},\qquad
\eta_{\rm code} = -\frac{\ddot\phi}{H\dot\phi},\qquad
\varepsilon_2 = \frac{d\ln\varepsilon_1}{dN} = -2\,\eta_{\rm code} + 2\,\varepsilon_1$$

**USR limit.** When the potential is extremely flat ($V' \approx 0$), the Klein-Gordon equation reduces to $\ddot\phi \approx -3H\dot\phi$, giving $\eta_{\rm code} \approx +3$ and consequently $\varepsilon_2 \approx -6$. Under these conditions $\varepsilon_1 \propto a^{-6}$, meaning the first slow-roll parameter plummets and the power spectrum develops a characteristic dip.


In [ ]:
def run_bg(model_cfg):
    model = HiggsModel(lam=model_cfg['lam'], xi=model_cfg['xi'])
    model.phi0 = model_cfg['phi0']
    model.yi = model_cfg['yi']
    T_span = np.linspace(0, 5000, 10000)
    sol = run_background_simulation(model, T_span)
    derived = get_derived_quantities(sol, model)
    return sol, derived, model, T_span

sol_usr, der_usr, mod_usr, T_usr = run_bg(FIELD_CONFIG)
sol_sr,  der_sr,  mod_sr,  T_sr  = run_bg(SR_CONFIG)

def compute_eps2(epsH, N):
    return np.gradient(np.log(np.maximum(epsH, 1e-30)), N)

N_usr = der_usr['N']; eps1_usr = der_usr['epsH']; eta_code_usr = der_usr['etaH']
N_sr  = der_sr['N'];  eps1_sr  = der_sr['epsH'];  eta_code_sr  = der_sr['etaH']

eps2_usr = compute_eps2(eps1_usr, N_usr)
eps2_sr  = compute_eps2(eps1_sr,  N_sr)

# mask inflation (eps1 < 1)
mask_usr = eps1_usr < 1
mask_sr  = eps1_sr  < 1

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: eps1(N)
ax1.semilogy(N_usr[mask_usr], eps1_usr[mask_usr], 'r-', lw=2, label=f'USR (yi={FIELD_CONFIG["yi"]})')
ax1.semilogy(N_sr[mask_sr],   eps1_sr[mask_sr],   'b-', lw=2, label=f'SR  (yi={SR_CONFIG["yi"]})')
ax1.axhline(1.0, color='k', ls=':', alpha=0.4, label='$\\varepsilon_1 = 1$')
ax1.set_xlabel('N [e-folds]'); ax1.set_ylabel('$\\varepsilon_1$')
ax1.set_title('First Hubble flow parameter $\\varepsilon_1$')
ax1.legend(); ax1.grid(True, alpha=0.3)

# Right: eps2(N) and eta_code(N)
ax2.plot(N_usr[mask_usr], eps2_usr[mask_usr], 'r-', lw=2, label='$\\varepsilon_2$ (USR)')
ax2.plot(N_sr[mask_sr],   eps2_sr[mask_sr],   'b-', lw=2, label='$\\varepsilon_2$ (SR)')
ax2.plot(N_usr[mask_usr], eta_code_usr[mask_usr], 'r--', lw=1.5, alpha=0.5, label='$\\eta_{\\rm code}$ (USR)')
ax2.axhline(-6, color='red', ls=':', alpha=0.5, label='$\\varepsilon_2 = -6$ (USR)')
ax2.axhline(0, color='k', ls='-', lw=0.5, alpha=0.3)
ax2.set_xlabel('N [e-folds]'); ax2.set_ylabel('$\\varepsilon_2,\\ \\eta_{\\rm code}$')
ax2.set_title('Second flow parameter $\\varepsilon_2 = d\\ln\\varepsilon_1/dN$')
ax2.legend(fontsize=9); ax2.grid(True, alpha=0.3)
ax2.set_ylim(-8, 4)

plt.tight_layout()
plt.show()


## Why USR suppresses $\mathcal{P}_{\mathcal{R}}(k)$

The Mukhanov-Sasaki equation for the curvature perturbation $v_k = z\,\mathcal{R}_k$ is:

$$v_k'' + \Bigl(k^2 - \frac{z''}{z}\Bigr)v_k = 0,\qquad
z = a\sqrt{2\varepsilon_1}$$

During USR, $\varepsilon_1 \propto a^{-6}$, dropping rapidly. This causes the effective mass term $z''/z$ to develop a sharp negative feature at the transition between SR and USR phases. Modes crossing the horizon near this transition experience a transient enhancement of the effective potential barrier, suppressing their frozen-in amplitude.

The result is a **localized dip** in $\mathcal{P}_{\mathcal{R}}(k)$ at the scales corresponding to the USR phase. The numerical pipeline below solves the full Mukhanov-Sasaki system for each k-mode and computes the resulting power spectrum.


In [ ]:
OUT_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', 'outputs/cmb_results/pspectra'))
os.makedirs(OUT_DIR, exist_ok=True)

def run_or_load(model_cfg, num_cfg, force_recompute=False):
    phi0 = model_cfg['phi0']; yi = model_cfg['yi']; xi = model_cfg['xi']
    model = HiggsModel(lam=model_cfg['lam'], xi=xi)
    model.phi0 = phi0; model.yi = yi
    # check cache
    if not force_recompute:
        for fname in sorted(os.listdir(OUT_DIR)):
            if not fname.endswith('.json'): continue
            with open(os.path.join(OUT_DIR, fname)) as f:
                meta = json.load(f)['metadata']
            if (abs(meta['phi0'] - phi0) < 1e-6 and abs(meta['yi'] - yi) < 1e-6
                and abs(meta.get('xi', 0) - xi) < 1e-6 and meta.get('N_star', meta.get('N_pivot')) == num_cfg['N_star']):
                print(f'Loaded cached: {fname}')
                return load_pspectrum(os.path.join(OUT_DIR, fname))
    # run
    k_grid = build_weighted_kgrid(num_cfg['k_min'], num_cfg['k_max'], num_cfg['k_pivot_phys']) if num_cfg['use_weighted'] else None
    print(f'Computing P_R(k) for phi0={phi0}, yi={yi}...')
    result = run_pspectrum_pipeline(
        model=model, phi0=phi0, yi=yi,
        k_min=num_cfg['k_min'], k_max=num_cfg['k_max'],
        num_k=num_cfg['num_k'], k_pivot_phys=num_cfg['k_pivot_phys'],
        N_star=num_cfg['N_star'], normalize_to_As=True, As=num_cfg['As'],
        k_phys_grid=k_grid,
    )
    if result['status'] != 'success':
        raise RuntimeError(f'Pipeline failed: {result.get("message", "unknown")}')
    data = load_pspectrum(result['output_file'])
    print(f'Done -> {os.path.basename(result["output_file"])}')
    return data

data_usr = run_or_load(FIELD_CONFIG, NUM_PARAMS)
data_sr  = run_or_load(SR_CONFIG,    NUM_PARAMS)

k_usr = data_usr['k_phys'];  PS_usr = data_usr['P_S'];  meta_usr = data_usr['metadata']
k_sr  = data_sr['k_phys'];   PS_sr  = data_sr['P_S'];   meta_sr  = data_sr['metadata']
As = meta_usr['As_target']

print(f'USR: N_total = {meta_usr["N_total"]:.2f}, N_star = {meta_usr.get("N_star", meta_usr.get("N_pivot"))}')
print(f'SR:  N_total = {meta_sr["N_total"]:.2f},  N_star = {meta_sr.get("N_star", meta_sr.get("N_pivot"))}')


In [ ]:
def local_ns(k, PS):
    dlnP = np.gradient(np.log(PS))
    dlnk = np.gradient(np.log(k))
    return 1 + dlnP / dlnk

ns_usr = local_ns(k_usr, PS_usr)
ns_sr  = local_ns(k_sr,  PS_sr)

# Locate dip
mask_dip = k_usr < 1.0
i_dip = np.argmin(PS_usr[mask_dip])
k_dip = k_usr[mask_dip][i_dip]
PS_dip = PS_usr[mask_dip][i_dip]

fig = plt.figure(figsize=(15, 10))
gs = GridSpec(2, 2, figure=fig, hspace=0.3, wspace=0.3)

ax1 = fig.add_subplot(gs[0, 0])
ax1.loglog(k_usr, PS_usr, 'r-', lw=2, label=f'USR (yi={FIELD_CONFIG["yi"]})')
ax1.loglog(k_sr,  PS_sr,  'b-', lw=2, label=f'SR  (yi={SR_CONFIG["yi"]})')
ax1.axhline(As, color='grey', ls='--', lw=1, alpha=0.5, label='Scale-invariant')
PS_lcdm = As * (k_usr / meta_usr['k_pivot_phys'])**(0.965 - 1)
ax1.loglog(k_usr, PS_lcdm, 'k--', lw=1.2, alpha=0.6, label='LCDM (n_s=0.965)')
ax1.axvline(meta_usr['k_pivot_phys'], color='gray', ls=':', alpha=0.5, label='pivot k=0.05')
ax1.axvspan(1e-4, 1e-2, color='orange', alpha=0.06)
ax1.annotate(f'Dip: k={k_dip:.2e}', xy=(k_dip, PS_dip),
             xytext=(k_dip*0.3, PS_dip*0.6), arrowprops=dict(arrowstyle='->', color='darkred'),
             fontsize=9, color='darkred')
ax1.set_xlabel('k [Mpc⁻¹]'); ax1.set_ylabel('$\\mathcal{P}_{\\mathcal{R}}(k)$')
ax1.set_title('Primordial Power Spectrum')
ax1.legend(fontsize=8); ax1.grid(True, alpha=0.3); ax1.set_xlim(1e-5, 2)

ax2 = fig.add_subplot(gs[0, 1])
ax2.semilogx(k_usr, ns_usr, 'r-', lw=1.5, label='USR')
ax2.semilogx(k_sr,  ns_sr,  'b-', lw=1.5, label='SR')
ax2.axhline(1.0, color='k', ls=':', alpha=0.4)
ax2.axhline(0.965, color='gray', ls='--', alpha=0.5, label='Planck n_s=0.965')
ax2.set_xlabel('k [Mpc⁻¹]'); ax2.set_ylabel('$n_s(k)$')
ax2.set_title('Local Spectral Index $n_s(k)$')
ax2.legend(fontsize=8); ax2.grid(True, alpha=0.3); ax2.set_xlim(1e-5, 2)

ax3 = fig.add_subplot(gs[1, :])
loss_si   = (PS_usr / As - 1) * 100
loss_lcdm = (PS_usr / PS_lcdm - 1) * 100
loss_sr   = (PS_usr / np.interp(k_usr, k_sr, PS_sr) - 1) * 100
ax3.semilogx(k_usr, loss_si,   color='grey', ls='--', lw=1.2, label='vs Scale-invariant')
ax3.semilogx(k_usr, loss_lcdm, color='k',    ls='-',  lw=1.5, label='vs LCDM (n_s=0.965)')
ax3.semilogx(k_usr, loss_sr,   color='blue', ls='-.', lw=1.2, alpha=0.6, label='vs SR Higgs')
ax3.axhline(0, color='k', lw=0.5, alpha=0.5)
ax3.axvspan(1e-4, 1e-2, color='orange', alpha=0.06)
ax3.set_xlabel('k [Mpc⁻¹]'); ax3.set_ylabel('Power loss [%]')
ax3.set_title('Power Loss of USR vs Reference Spectra')
ax3.legend(fontsize=8); ax3.grid(True, alpha=0.3); ax3.set_xlim(1e-5, 2)

plt.suptitle(f'USR: phi0={FIELD_CONFIG["phi0"]}, yi={FIELD_CONFIG["yi"]}, xi={FIELD_CONFIG["xi"]}',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


## Dip interpretation

The USR dip manifests at $k \approx $ determined by the $\phi_0$ and $y_i$ parameters. The dip center corresponds to modes that crossed the horizon just as the USR phase began.

Key observations from the diagnostic plots:

- The dip in $\mathcal{P}_{\mathcal{R}}(k)$ can reach several tens of percent relative to the LCDM baseline, depending on the strength ($|y_i|$) and duration of the USR phase.
- The spectral index $n_s(k)$ rises above unity on the recovery side of the dip — this is a characteristic "ringing" signature of USR transitions.
- Scales corresponding to $k \sim 10^{-4} - 10^{-2}\, \mathrm{Mpc}^{-1}$ map to $\ell = k \cdot r_{\rm ls}$ with $r_{\rm ls} \approx 14000\,\mathrm{Mpc}$, i.e., $\ell \sim 1.4 - 140$. The low-$\ell$ deficit observed by Planck ($\ell \lesssim 30$) corresponds to the largest scales, $k \lesssim 2\times10^{-3}\, \mathrm{Mpc}^{-1}$.

Whether the dip is sufficient to explain the Planck low-$\ell$ anomaly depends on the Sachs-Wolfe $C_\ell$ calculation below.


## Sachs-Wolfe $C_\ell$

On large angular scales ($\ell \lesssim 30$) the dominant contribution to the CMB temperature anisotropy is the Sachs-Wolfe effect:

$$C_\ell^{TT} \approx \frac{4\pi}{25} \int \frac{dk}{k}\, \mathcal{P}_{\mathcal{R}}(k)\, j_\ell^2(k\,r_{\rm ls})$$

The factor $4\pi/25$ arises from the transfer function for the Sachs-Wolfe plateau: $\Delta_\ell^{\rm SW}(k) = \frac15 j_\ell(k\,r_{\rm ls})$. The comoving distance to last scattering is $r_{\rm ls} \approx 14000\,\mathrm{Mpc}$.

The angular power spectrum is conventionally reported as

$$D_\ell = \frac{\ell(\ell+1)}{2\pi}C_\ell \times (T_{\rm cmb}\times10^6)^2 \quad[\mu{\rm K}^2]$$

with $T_{\rm cmb} = 2.7255\,\mathrm{K}$. This conversion is applied below for comparison with Planck 2018 binned data.


In [ ]:
r_ls = 14000.0
ell_max = 29

ells_usr, CTT_usr = compute_cl_sw(data_usr, ell_max=ell_max, r_ls=r_ls)
ells_sr,  CTT_sr  = compute_cl_sw(data_sr,  ell_max=ell_max, r_ls=r_ls)
ells_lcdm, CTT_lcdm, _ = compute_cl_sw_powerlaw(
    k_min=k_usr[0], k_max=k_usr[-1], As=As, ns=0.965, ell_max=ell_max, r_ls=r_ls
)

ells_pl, D_pl, D_err = get_planck_data()
mask_pl = ells_pl <= ell_max
ells_pl = ells_pl[mask_pl]; D_pl = D_pl[mask_pl]; D_err = D_err[mask_pl]

Tcmb = 2.7255
conv = (Tcmb * 1e6)**2
D_usr  = CTT_usr  * ells_usr  * (ells_usr + 1)  * conv / (2*np.pi)
D_sr   = CTT_sr   * ells_sr   * (ells_sr + 1)   * conv / (2*np.pi)
D_lcdm = CTT_lcdm * ells_lcdm * (ells_lcdm + 1) * conv / (2*np.pi)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: D_ell vs Planck
ax = axes[0]
ax.errorbar(ells_pl, D_pl, yerr=D_err, fmt='ko', ms=5, capsize=3, label='Planck 2018')
ax.semilogx(ells_usr,  D_usr,  'r-o', ms=4, lw=2, label='USR Higgs')
ax.semilogx(ells_sr,   D_sr,   'b-s', ms=4, lw=2, label='SR Higgs')
ax.semilogx(ells_lcdm, D_lcdm, 'grey', ls='--', lw=1.5, label='LCDM (ns=0.965)')
ax.set_xlabel('$\\ell$'); ax.set_ylabel('$D_\\ell$ [$\\mu$K$^2$]')
ax.set_title('CMB Temperature Power Spectrum $D_\\ell$')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3); ax.set_xlim(2, 30)

# Panel 2: Ratio
ax = axes[1]
ratio_usr  = D_usr  / np.interp(ells_usr,  ells_lcdm, D_lcdm)
ratio_sr   = D_sr   / np.interp(ells_sr,   ells_lcdm, D_lcdm)
ax.semilogx(ells_usr, ratio_usr, 'r-o', ms=5, lw=1.5, label='USR / LCDM')
ax.semilogx(ells_sr,  ratio_sr,  'b-s', ms=5, lw=1.5, label='SR / LCDM')
ax.axhline(1.0, color='k', ls='--', lw=1, alpha=0.5)
ax.fill_between(ells_usr, ratio_usr, 1.0, alpha=0.15, color='red', where=(ratio_usr < 1.0))
ax.set_xlabel('$\\ell$'); ax.set_ylabel('$D_\\ell / D_\\ell^{\\rm LCDM}$')
ax.set_title('Suppression vs LCDM')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3); ax.set_xlim(2, 30)

# Panel 3: Pulls
ax = axes[2]
D_usr_pl  = np.interp(ells_pl, ells_usr,  D_usr)
D_sr_pl   = np.interp(ells_pl, ells_sr,   D_sr)
D_lcdm_pl = np.interp(ells_pl, ells_lcdm, D_lcdm)
pull_usr  = (D_usr_pl  - D_pl) / D_err
pull_sr   = (D_sr_pl   - D_pl) / D_err
pull_lcdm = (D_lcdm_pl - D_pl) / D_err
ax.bar(ells_pl - 0.3, pull_usr,  width=0.25, color='red',  alpha=0.7, label='USR')
ax.bar(ells_pl,       pull_sr,   width=0.25, color='blue', alpha=0.7, label='SR')
ax.bar(ells_pl + 0.3, pull_lcdm, width=0.25, color='gray', alpha=0.5, label='LCDM')
ax.axhline(0, color='k', lw=0.5); ax.axhline(1, color='k', ls=':', lw=0.5, alpha=0.3)
ax.axhline(-1, color='k', ls=':', lw=0.5, alpha=0.3)
ax.set_xlabel('$\\ell$'); ax.set_ylabel('Pull (model $-$ data) / $\\sigma$')
ax.set_title('Per-$\\ell$ Pulls')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3, axis='y'); ax.set_xlim(1, 31)

plt.suptitle(f'CMB Low-ell: phi0={FIELD_CONFIG["phi0"]}, yi={FIELD_CONFIG["yi"]}, xi={FIELD_CONFIG["xi"]}',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## C_\ell trajectory check

The Sachs-Wolfe $C_\ell$ comparison reveals:

- The USR model brings $D_\ell$ closer to the low Planck data points at $\ell \approx 10$--$20$ compared to LCDM, reducing the systematic offset.
- The suppression ratio $D_\ell^{\rm USR}/D_\ell^{\rm LCDM}$ dips below unity at precisely the multipoles most discrepant in Planck.
- The pull chart shows that negative pulls (model below data) for LCDM are reduced by the USR model at most low $\ell$.

**Important caveat:** This is the Sachs-Wolfe approximation only ($\ell \lesssim 30$). A full Boltzmann calculation (e.g., CAMB) would refine the high-$\ell$ tail and include the ISW and Doppler contributions. However, for $\ell \lesssim 30$ the SW term dominates, so the qualitative conclusions are robust.


## Summary metrics

Below is a consolidated comparison of key quantities across the three scenarios.


In [ ]:
# Compute summary metrics
k_dip_val = k_dip
PS_dip_val = PS_dip
loss_at_dip_lcdm = loss_lcdm[mask_dip][i_dip] if 'loss_lcdm' in dir() else float('nan')

ns_usr_pivot = np.interp(meta_usr['k_pivot_phys'], k_usr, ns_usr)
ns_sr_pivot  = np.interp(meta_sr['k_pivot_phys'],  k_sr,  ns_sr)

PS_usr_at_pivot = np.interp(meta_usr['k_pivot_phys'], k_usr, PS_usr) / As
PS_sr_at_pivot  = np.interp(meta_sr['k_pivot_phys'],  k_sr,  PS_sr)  / As

print(f'{"Metric":<35s} {"USR":>12s} {"SR":>12s} {"LCDM":>12s}')
print('-' * 71)
print(f'{"k_dip [Mpc-1]":<35s} {k_dip_val:>12.2e} {"---":>12s} {"---":>12s}')
print(f'{"P_R(k_dip)/A_s":<35s} {PS_dip_val/As:>12.4f} {"---":>12s} {"---":>12s}')
print(f'{"DeltaP/P vs LCDM at dip [%]":<35s} {loss_at_dip_lcdm:>12.1f} {"---":>12s} {"---":>12s}')
print(f'{"n_s at pivot":<35s} {ns_usr_pivot:>12.4f} {ns_sr_pivot:>12.4f} {"0.9650":>12s}')
print(f'{"P_R(k0)/A_s at pivot":<35s} {PS_usr_at_pivot:>12.4f} {PS_sr_at_pivot:>12.4f} {"1.0000":>12s}')


## Conclusion

**What USR does:** A transient USR phase drives $\varepsilon_2 \to -6$ for ~1–2 e-folds. This creates a localized feature in the Mukhanov-Sasaki effective mass, suppressing $\mathcal{P}_{\mathcal{R}}(k)$ on the corresponding scales and reducing the predicted low-$\ell$ CMB power.

**Main tunables:**
- $\phi_0$: controls the total number of e-folds and where in k-space the USR feature sits.
- $y_i$ (initial velocity): controls the strength/depth of the USR phase. More negative $\rightarrow$ stronger dip.

**Key results:**
- The USR dip can reach significant power loss (tens of percent) on CMB scales.
- The Sachs-Wolfe $C_\ell$ shows visible improvement over LCDM at $\ell \lesssim 20$.
- The characteristic rise in $n_s(k)$ above 1 post-dip is a testable signature.

**Caveats:**
1. Sachs-Wolfe approximation only — full CAMB integration would refine the comparison.
2. Single-field inflationary model — multi-field or modified gravity alternatives also exist.
3. Fine-tuning required: the USR phase must be positioned at the right scale and have appropriate duration.
4. Non-Gaussianity constraints: USR typically produces large $f_{\rm NL}$, which may be in tension with Planck bispectrum bounds.

**Next steps:**
- Full CAMB integration for $\ell$ up to 2500.
- Cross-check with `FullHiggsModel` (exact conformal inversion).
- Compute $f_{\rm NL}^{\rm local}$ for the USR configuration.
- Scan the ($\phi_0, y_i, \xi$) parameter space to quantify the viable region.
